In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "Python" | "JSON" | "Regex",
        "solution_criteria": "The key criteria for a correct solution to the task"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects as a pure JSON array with no Markdown formatting.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
import json

dataset = generate_dataset()

with open("files/01-prompt-evals-dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "solution_criteria": A number between 1-10 indicating how well the solution meets the criteria
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "solution_criteria": number,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [6]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [7]:
def run_prompt(test_case):
    """Merges the prompt with the test case input, then returns the result."""
    prompt = f"""
                Please solve the following task: 
                {test_case['task']}
                
                * Respond only in {test_case['format']} format. 
                * Do not include any explanations or additional text.
            """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, f"```{test_case['format']}")
    return chat(messages)

def run_test_case(test_case):
    """Calls the `run_prompt` function, then grades the result against the expected output."""
    
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    solution_criteria = model_grade["solution_criteria"]
    
    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "solution_criteria": solution_criteria
    }
    
from statistics import mean
    
def run_eval(dataset):
    """Loads the dataset and calls `run_test_case` for each test case."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
        
    return results

In [8]:
with open("files/01-prompt-evals-dataset.json", "r") as f:
    dataset = json.load(f)
    
    results = run_eval(dataset)

Average score: 6.5


In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n^s3://([a-z0-9.-]+)/\n```",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and region from an S3 URI in the format s3://bucket-name/key and extract just the bucket name",
      "format": "Regex",
      "solution_criteria": "The regex pattern should correctly extract the bucket name from various S3 URIs like s3://my-bucket/path/to/file and s3://another-bucket-123/file.txt, ignoring the region and key path"
    },
    "score": 8.5,
    "reasoning": "The regex successfully extracts bucket names from standard S3 URIs like 's3://my-bucket/path' and 's3://another-bucket-123/file.txt' by capturing the alphanumeric and dot/dash characters. However, it has a practical limitation: S3 bucket names are case-insensitive and can contain uppercase letters in the URI string, which this pattern would fail to match. The pattern itself is functionally correct for the most common use case but lacks robustness for real-world variations.",
    "solution_criteria": 7
  }